<a href="https://colab.research.google.com/github/Jyoti-Yadav2/R-for-bioinformatics/blob/main/Hidden_Markov_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#install R base
#Configure rpy2

In [2]:
install.packages("BiocManager")
BiocManager::install("seqinr")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Warning message:
“package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'seqinr'”
Old packages: 'callr', 'dbplyr', 'httr2', 'openssl', 'pak', 'pkgload', 'Rcpp',
  'rlang', 'rstudioapi', 'selectr', 'sessioninfo', 'shiny', 'tinytex', 'withr',
  'xfun', 'xml2', 'xtable', 'zip'



Multinomial model

In [3]:
nucleotides <- c("A","C","G","T")
probabilities1 <- c(0.2,0.3,0.3,0.2)
seqlength <-30
sample(nucleotides,seqlength,rep=TRUE,prob=probabilities1)

[1] "G" "A" "G" "T" "C" "C" "C" "C" "C" "C" "T" "G" "G" "T" "C" "A" "A" "C" "A"
[20] "C" "A" "T" "A" "A" "T" "C" "G" "A" "C" "T"

In [4]:
probabilities2 <- c(0.1,0.41,0.39,0.1)
sample(nucleotides,seqlength,rep=TRUE,prob = probabilities2)

[1] "G" "G" "G" "A" "C" "C" "G" "G" "C" "C" "G" "T" "C" "G" "C" "G" "C" "A" "C"
[20] "T" "G" "C" "C" "G" "G" "C" "C" "C" "C" "C"

Markov model

In [5]:
nucleotides <- c("A","C","G","T")
afterAprobs <- c(0.2,0.3,0.3,0.2)
afterCprobs <- c(0.1,0.41,0.39,0.1)
afterGprobs <- c(0.25,0.25,0.25,0.25)
afterTprobs <- c(0.5, 0.17, 0.17, 0.17)
mytransitionmatrix <- matrix(c(afterAprobs, afterCprobs, afterGprobs,
                                 afterTprobs), 4, 4, byrow = TRUE) # Create a 4 x 4 matrix
rownames(mytransitionmatrix) <- nucleotides
colnames(mytransitionmatrix) <- nucleotides
mytransitionmatrix

,A,C,G,T
A,0.20,0.30,0.30,0.20
C,0.10,0.41,0.39,0.10
G,0.25,0.25,0.25,0.25
T,0.50,0.17,0.17,0.17


In [6]:
generatemarkovseq <- function(transitionmatrix, initialprobs, seqlength)
{
nucleotides <- c("A", "C", "G", "T") # Define the alphabet of nucleotides
mysequence <- character() # Create a vector for storing the new sequence
# Choose the nucleotide for the first position in the sequence:
firstnucleotide <- sample(nucleotides, 1, rep=TRUE, prob=initialprobs)
mysequence[1] <- firstnucleotide # Store the nucleotide for the first position of the sequence
for (i in 2:seqlength)
{
prevnucleotide <- mysequence[i-1] # Get the previous nucleotide in the new sequence
# Get the probabilities of the current nucleotide, given previous nucleotide"prevnucleotide":
probabilities <- transitionmatrix[prevnucleotide,]
# Choose the nucleotide at the current position of the sequence:
nucleotide <- sample(nucleotides, 1, rep=TRUE, prob=probabilities)
mysequence[i] <- nucleotide # Store the nucleotide for the current position of the sequence
}
return(mysequence)
}

In [7]:
myinitialprobs <- c(0.25, 0.25, 0.25, 0.25)
generatemarkovseq(mytransitionmatrix, myinitialprobs, 30)

[1] "T" "A" "G" "C" "A" "C" "C" "C" "C" "T" "A" "C" "G" "C" "G" "C" "G" "C" "G"
[20] "A" "A" "G" "T" "A" "C" "G" "C" "C" "C" "G"

In [8]:
states <- c("AT-rich", "GC-rich") # Define the names of the states
ATrichprobs <- c(0.7, 0.3) # Set the probabilities of switching states, where the previous state was "AT-rich"
GCrichprobs <- c(0.1, 0.9) # Set the probabilities of switching states, where the previous state was "GC-rich"
thetransitionmatrix <- matrix(c(ATrichprobs, GCrichprobs), 2, 2, byrow = TRUE)
# Create a 2 x 2 matrix
rownames(thetransitionmatrix) <- states
colnames(thetransitionmatrix) <- states
thetransitionmatrix

,AT-rich,GC-rich
AT-rich,0.7,0.3
GC-rich,0.1,0.9


In [9]:
nucleotides <- c("A","C","G","T")
ATrichstateprobs <- c(0.39, 0.1, 0.1, 0.41) # Set the values of probabilities, for the AT-rich state
GCrichstateprobs <- c(0.1, 0.41, 0.39, 0.1) # Set the values of probabilities, for the GC-rich state
theemissionmatrix <- matrix(c(ATrichstateprobs, GCrichstateprobs), 2, 4, byrow =TRUE)
# Create a 2 x 4 matrix
rownames(theemissionmatrix) <- states
colnames(theemissionmatrix) <- nucleotides
theemissionmatrix

,A,C,G,T
AT-rich,0.39,0.10,0.10,0.41
GC-rich,0.10,0.41,0.39,0.10


In [10]:
# Function to generate a DNA sequence, given a HMM and the length of sequence to be generated.
generatehmmseq <- function(transitionmatrix, emissionmatrix, initialprobs, seqlength)
{
  nucleotides <- c("A", "C", "G", "T") # Define the alphabet of nucleotides
  states <- c("AT-rich", "GC-rich") # Define the names of the states
  mysequence <- character()
  # Create a vector for storing new sequence
  mystates <- character()
  # Create a vector for storing state that each position in the new sequence
  # was generated by
  # Choose the state for the first position in the sequence:
  firststate <- sample(states, 1, rep=TRUE, prob=initialprobs)
  # Get the probabilities of the current nucleotide, given that we are in state "firststate":
  probabilities <- emissionmatrix[firststate,]
  # Choose the nucleotide for the first position in the sequence:
  firstnucleotide <- sample(nucleotides, 1, rep=TRUE, prob=probabilities)
  mysequence[1] <- firstnucleotide
  # Store the nucleotide for first position of the sequence
  mystates[1] <- firststate
  # Store the state that the first position in the sequence was generated by
  for (i in 2:seqlength)
  {
    prevstate <- mystates[i-1]
    # Get the state that the previous nucleotide in the sequence was generated by
    # Get the probabilities of the current state, given that the previous nucleotide was generated by state "prevstate"
    stateprobs <- transitionmatrix[prevstate,]
    # Choose the state for the ith position in the sequence:
    state <- sample(states, 1, rep=TRUE, prob=stateprobs)
    # Get the probabilities of the current nucleotide, given that we are in the state "state":
    probabilities <- emissionmatrix[state,]
    # Choose the nucleotide for the ith position in the sequence:
    nucleotide <- sample(nucleotides, 1, rep=TRUE, prob=probabilities)
    mysequence[i] <- nucleotide # Store the nucleotide for current position of the sequence
    mystates[i] <- state # Store the state that the current position in the sequence was generated by
    }
    for (i in 1:length(mysequence))
    {
      nucleotide <- mysequence[i]
      state <- mystates[i]
      print(paste("Position", i, ", State", state, ", Nucleotide = ",nucleotide))
      }
    }

In [11]:
theinitialprobs <- c(0.5, 0.5)
generatehmmseq(thetransitionmatrix, theemissionmatrix, theinitialprobs, 30)

[1] "Position 1 , State GC-rich , Nucleotide =  G"
[1] "Position 2 , State GC-rich , Nucleotide =  G"
[1] "Position 3 , State GC-rich , Nucleotide =  C"
[1] "Position 4 , State GC-rich , Nucleotide =  G"
[1] "Position 5 , State GC-rich , Nucleotide =  A"
[1] "Position 6 , State GC-rich , Nucleotide =  G"
[1] "Position 7 , State GC-rich , Nucleotide =  C"
[1] "Position 8 , State GC-rich , Nucleotide =  A"
[1] "Position 9 , State GC-rich , Nucleotide =  G"
[1] "Position 10 , State GC-rich , Nucleotide =  G"
[1] "Position 11 , State AT-rich , Nucleotide =  G"
[1] "Position 12 , State AT-rich , Nucleotide =  T"
[1] "Position 13 , State AT-rich , Nucleotide =  A"
[1] "Position 14 , State GC-rich , Nucleotide =  C"
[1] "Position 15 , State GC-rich , Nucleotide =  G"
[1] "Position 16 , State GC-rich , Nucleotide =  C"
[1] "Position 17 , State GC-rich , Nucleotide =  A"
[1] "Position 18 , State AT-rich , Nucleotide =  G"
[1] "Position 19 , State AT-rich , Nucleotide =  A"
[1] "Position 20 , St

In [12]:
viterbi <- function(sequence, transitionmatrix, emissionmatrix)
# This carries out the Viterbi algorithm.
# ( cran.r-project.org/doc/contrib/Krijnen-IntroBioInfStatistics.pdf )
{
  # Get the names of the states in the HMM:
  states <- rownames(theemissionmatrix)
  # Make the Viterbi matrix v:
  v <- makeViterbimat(sequence, transitionmatrix, emissionmatrix)
  # Go through each of the rows of the matrix v (where each row represents
  # a position in the DNA sequence), and find out which column has the
  # maximum value for that row (where each column represents one state of
  # the HMM):
  mostprobablestatepath <- apply(v, 1, function(x) which.max(x))
  # Print out the most probable state path:
  prevnucleotide <- sequence[1]
  prevmostprobablestate <- mostprobablestatepath[1]
  prevmostprobablestatename <- states[prevmostprobablestate]
  startpos <- 1
  for (i in 2:length(sequence))
  {
    nucleotide <- sequence[i]
    mostprobablestate <- mostprobablestatepath[i]
    mostprobablestatename <- states[mostprobablestate]
    if (mostprobablestatename != prevmostprobablestatename)
    {
      print(paste("Positions",startpos,"-",(i-1), "Most probable state = ", prevmostprobablestatename))
      startpos <- i
      }
      prevnucleotide <- nucleotide
      prevmostprobablestatename <- mostprobablestatename
      }
      print(paste("Positions",startpos,"-",i, "Most probable state = ",prevmostprobablestatename))
      }


In [13]:
makeViterbimat <- function(sequence, transitionmatrix, emissionmatrix)
# This makes the matrix v using the Viterbi algorithm.
# ( cran.r-project.org/doc/contrib/Krijnen-IntroBioInfStatistics.pdf )
{
  # Change the sequence to uppercase
  sequence <- toupper(sequence)
  # Find out how many states are in the HMM
  numstates <- dim(transitionmatrix)[1]
  # Make a matrix with as many rows as positions in the sequence, and as many
  # columns as states in the HMM
  v <- matrix(NA, nrow = length(sequence), ncol = dim(transitionmatrix)[1])
  # Set the values in the first row of matrix v (representing first position of the sequence) to 0
  v[1, ] <- 0
  # Set the value in the first row of matrix v, first column to 1
  v[1,1] <- 1
  # Fill in the matrix v:
  for (i in 2:length(sequence)) # For each position in the DNA sequence:
  {
    for (l in 1:numstates) # For each of the states of in the HMM:
    {
      # Find the probabilility, if we are in state l, of choosing nucleotide at position in the sequence
      statelprobnucleotidei <- emissionmatrix[l,sequence[i]]
      # v[(i-1),] gives the values of v for the (i-1)th row of v, ie. the (i-1)th position in the sequence.
      # In v[(i-1),] there are values of v at the (i-1)th row of the sequence for each possible state k.
      # v[(i-1),k] gives the value of v at the (i-1)th row of the sequence for a particular state k.
      # transitionmatrix[l,] gives the values in the lth row of transition matrix, xx should not be transitionmatrix[,l]?
      # probabilities of changing from a previous state k to a current state l.
      # max(v[(i-1),] * transitionmatrix[l,]) is the maximum probability for nucleotide observed
      # at the previous position in the sequence in state k, followed by a transition from previous
      # state k to current state l at the current nucleotide position.
      # Set the value in matrix v for row i (nucleotide position i), column l (state l) to be:
      v[i,l] <- statelprobnucleotidei * max(v[(i-1),] * transitionmatrix[,l])
      }
    }
    return(v)
  }

In [14]:
myseq <- c("A", "A", "G", "C", "G", "T", "G", "G", "G", "G", "C", "C", "C", "C",
           "G", "G", "C", "G", "A", "C", "A", "T", "G", "G", "G", "G", "T", "G", "T", "C")
viterbi(myseq, thetransitionmatrix, theemissionmatrix)

[1] "Positions 1 - 2 Most probable state =  AT-rich"
[1] "Positions 3 - 21 Most probable state =  GC-rich"
[1] "Positions 22 - 22 Most probable state =  AT-rich"
[1] "Positions 23 - 30 Most probable state =  GC-rich"
